<!-- cabecera-entorno -->
## Antes de empezar

**Clase 10 · Visualización interactiva: Plotly + Streamlit** — Bloque 3 · Reto. Este cuaderno lo
recorre **usted solo**, leyendo: cada tarea trae la explicación y los comandos que necesita. El
profesor circula por el salón resolviendo dudas. Es el entregable de la clase.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `reto.ipynb` como
`reto_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `FileNotFoundError` al leer el CSV | El notebook se abrió desde otra carpeta, o falta hacer `git pull` | Manual, problema 6 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys
from pathlib import Path

try:
    import pandas as pd
    import plotly.express as px
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

RUTA_VERIFICACION = "../datos/educacion_estadisticas.csv"
if Path(RUTA_VERIFICACION).exists():
    print("Datos: encontrados en", RUTA_VERIFICACION)
else:
    print("FALTA el archivo", RUTA_VERIFICACION, "- abra en VSCode la carpeta raíz del curso",
          "y ejecute 'git pull'. Ver ../INSTALACION.md, problema 6.")

# Clase 10 · Reto — Del gráfico interactivo al dashboard

**Equipo:**

**Fecha:**

**Consigna completa:** `reto.md`

Este cuaderno tiene dos partes, y la segunda es el entregable real.

| Parte | Datos | Qué se hace | Cómo se comprueba |
|-------|-------|-------------|-------------------|
| **A · Calibración** (T1-T6) | `../datos/educacion_estadisticas.csv` | Los tres gráficos de Plotly sobre datos que **no** vio en el demo | Respuesta única: `comprobar` compara una huella de su resultado |
| **B · Su dashboard** (T7-T10) | El dataset **de su equipo** | El diseño, los datos, los KPIs y las figuras de su app | No hay respuesta única: se revisa la estructura |

**Por qué dos partes.** La parte A es media hora de práctica con datos ajenos: si algo de Plotly no
quedó claro en el demo, aquí se cae y se arregla rápido, con un verificador que dice si está bien.
La parte B es su proyecto, donde nadie puede decirle cuál es la respuesta correcta porque depende de
su dataset y de su pregunta.

**Y la parte B no es un ejercicio parecido al entregable del Momento 2: es el entregable del Momento
2.** Lo que empiece hoy es lo que sustenta en la clase 12.

## Cómo se recorre este cuaderno

| Parte de la tarea | Qué contiene |
|-------------------|--------------|
| **La pregunta** | Lo que hay que responder, escrito en español |
| **El concepto** | Qué técnica aplica y por qué esa y no otra |
| **Los comandos** | Las instrucciones que va a usar, escritas de forma genérica |
| **Lo que decide usted** | Qué columna, qué recorte, qué título. Ahí no hay respuesta escrita |
| **La celda de código** | Los pasos numerados en comentarios. Usted escribe las líneas |
| **La comprobación** | `comprobar('T1', ...)` dice si el resultado es el correcto, **sin mostrárselo** |

Las tareas de figura, de plan y de KPIs se comprueban distinto: no hay una única figura correcta, así
que lo que se revisa es lo verificable (que exista, que sea del tipo pedido, que tenga título y
unidades). **Que el título diga una conclusión verdadera lo juzga usted**, y es lo que más pesa.

---

# Parte A · Calibración con datos nuevos

## El dataset

`educacion_estadisticas.csv`: indicadores de educación básica y media por departamento y año, del
Ministerio de Educación vía datos.gov.co. 482 filas x 37 columnas, de 2011 a 2024. Es el mismo
archivo de la clase 3, así que ya sabe qué tiene de sucio.

| Columna que va a usar | Qué es |
|-----------------------|--------|
| `ano` | El año del reporte |
| `departamento` | El departamento. **Viene sucio**: mayúsculas, minúsculas, espacios de sobra y tildes inconsistentes (`Guainia` y `Guainía`). La celda de carga ya lo estandariza con la receta de la clase 3 |
| `desercion` | Porcentaje de estudiantes que abandonan el año escolar. **La medida de hoy** |
| `cobertura_neta` | Porcentaje de niños de 5 a 16 años matriculados en el nivel que les corresponde |
| `aprobacion` | Porcentaje de estudiantes que aprueban el año |

**El ancla, igual que la guía de la OMS del demo:** la meta del Plan Nacional de Desarrollo es
mantener la deserción por debajo del 3%. Sin ese número, decir "4,2%" no significa nada.

La celda de abajo ya está escrita: carga y aplica la limpieza mínima de la clase 3. Ejecútela y
**lea lo que imprime**.

In [ ]:
import pandas as pd
import plotly.express as px

# Este cuaderno vive en clase10/reto/, y el CSV dos carpetas más arriba, en datasets/
educacion = pd.read_csv("../datos/educacion_estadisticas.csv")

filas_crudas = len(educacion)

# Limpieza mínima, la de la clase 3, y solo lo que impide graficar:
# 1. El nombre del departamento viene con espacios sobrantes, con mayúsculas inconsistentes
#    ('  Nariño  ', 'NARIÑO', 'antioquia' son el mismo departamento y cuentan como tres) y con
#    tildes inconsistentes ('Guainia' y 'Guainía' también son el mismo). Es la receta de la
#    clase 3, completa: espacios, mayúsculas, tildes y la puntuación de 'BOGOTA, D,C,'.
#    Precio aceptado, el mismo de la clase 3: las etiquetas pierden la tilde ('Narino').
#    Un rótulo sin tilde es un detalle; una barra de más es una afirmación falsa.
educacion["departamento"] = (educacion["departamento"].astype(str)
                             .str.strip()
                             .str.normalize("NFKD")
                             .str.encode("ascii", "ignore")
                             .str.decode("utf-8")
                             .str.title()
                             .str.replace("D,C,", "D.C.", regex=False))

# 2. El año llega como float (2023.0). Como entero se ordena y se etiqueta bien en el eje.
educacion["ano"] = educacion["ano"].astype(int)

# 3. Sin deserción no hay nada que graficar. Se descartan esas filas, y se dice cuántas.
educacion = educacion.dropna(subset=["desercion", "departamento"]).copy()

META_DESERCION = 3.0

print(f"Filas crudas:     {filas_crudas}")
print(f"Filas de trabajo: {len(educacion)}  ({filas_crudas - len(educacion)} descartadas)")
print(f"Años: {educacion['ano'].min()} a {educacion['ano'].max()} | "
      f"Departamentos: {educacion['departamento'].nunique()} (deberían ser 33)")
print(f"Deserción: mínimo {educacion['desercion'].min():.2f}%, "
      f"máximo {educacion['desercion'].max():.2f}%, meta {META_DESERCION}%")

### El verificador

La celda de abajo define `comprobar(...)`, `comprobar_figura(...)`, `comprobar_plan(...)`,
`comprobar_kpis(...)`, `comprobar_datos(...)` y el punto de control. Ejecútela una vez y siga
adelante: es andamiaje del curso, no materia de la clase.

In [ ]:
# Verificador de las diez tareas. Ejecute esta celda una vez y siga adelante.
# No hace falta entenderla hoy: es andamiaje del curso, no materia de la clase.
import hashlib

import pandas as pd

_RESULTADOS = {}

_CLAVES = ["T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8", "T9", "T10"]

_PISTAS = {
    "T1": "Es groupby('ano', as_index=False) sobre educacion, con .mean() de 'desercion', y despues sort_values('ano'). Use el DataFrame limpio (educacion), no el crudo. Si le sobran filas o le salen NaN, agrupo antes de quitar los nulos de 'desercion'.",
    "T2": "Dos pasos: recorte educacion al ultimo ano (el maximo de la columna 'ano') y cuente cuantas filas tienen 'desercion' por encima de UMBRAL_DESERCION. (serie > umbral).sum() devuelve el conteo. Conviertalo a int.",
    "T3": "Es un groupby por 'departamento' con .mean() de 'desercion', ordenado de mayor a menor, y despues .head(10). Con as_index=False para que 'departamento' quede como columna. Deben quedar 10 filas exactas.",
    "T4": "px.line(serie_anual, x='ano', y='desercion', title=...) guardado en fig_linea. Falta el titulo con una conclusion, o la etiqueta del eje y: fig_linea.update_layout(yaxis_title='Desercion (%)').",
    "T5": "px.bar(top_departamentos, x=..., y=..., title=...) guardado en fig_barras. Ojo con el orden: la tabla ya viene ordenada, pero Plotly reordena solo si usted se lo pide. Falta el titulo largo o la etiqueta del eje de la medida.",
    "T6": "px.box(educacion, x='ano', y='desercion', title=...) guardado en fig_caja. El eje x es el ano: una caja por ano muestra la dispersion ENTRE departamentos, que es justamente lo que el promedio de T1 esconde.",
    "T7": "df_equipo tiene que ser un DataFrame ya cargado y ya renombrado: columnas en minuscula, sin tildes y sin espacios. Si el verificador se queja de las columnas, falta el .rename(columns={...}). Si se queja de que no hay numericas, falta pd.to_numeric(col, errors='coerce').",
    "T8": "plan_filtros es una lista de tres diccionarios con las claves 'columna', 'widget' y 'porque'. La columna tiene que existir en df_equipo con ese nombre exacto, tiene que haber al menos un multiselect y al menos un slider, y las tres columnas tienen que ser distintas entre si.",
    "T9": "calcular_kpis(df) tiene que calcular TODO a partir del df que recibe por parametro. Si algun KPI no cambia al recibir la mitad de las filas, es porque esta usando df_equipo por fuera en vez del parametro, o es una constante.",
    "T10": "figuras_equipo es una lista con sus tres figuras de Plotly. Cada una necesita title= con un mensaje (no una etiqueta) y la etiqueta del eje de la medida. Y las tres tienen que responder preguntas distintas: si las tres son de barras sobre la misma columna, cuentan como una."
}

_ESPERADO = {
    "T1": "e4d53158b0",
    "T2": "b77a824920",
    "T3": "339a706e31"
}


def _firma(valor):
    """Reduce un resultado a un texto reproducible, sin importar como se calculo."""
    if isinstance(valor, pd.DataFrame):
        partes = ["DataFrame", str(valor.shape), str([str(c) for c in valor.columns]),
                  str([str(i) for i in valor.index])]
        for columna in valor.columns:
            serie = valor[columna]
            if pd.api.types.is_bool_dtype(serie) or not pd.api.types.is_numeric_dtype(serie):
                partes.append(f"{columna}:{[str(v) for v in serie.tolist()]}")
            else:
                partes.append(f"{columna}:{round(float(serie.sum()), 4)}")
        return "|".join(partes)
    if isinstance(valor, pd.Series):
        return "|".join(["Series", str(len(valor)), str([str(i) for i in valor.index]),
                         str([str(v) for v in valor.tolist()])])
    if isinstance(valor, (list, tuple)):
        return "lista|" + "|".join(str(v) for v in valor)
    if not isinstance(valor, str):
        try:
            return f"numero|{round(float(valor), 4)}"
        except (TypeError, ValueError):
            pass
    return f"otro|{valor!r}"

def _huella(valor):
    return hashlib.sha256(_firma(valor).encode("utf-8")).hexdigest()[:10]

def _redondear(valor, decimales):
    if decimales is None or valor is None:
        return valor
    if isinstance(valor, (pd.DataFrame, pd.Series)):
        return valor.round(decimales)
    try:
        return round(float(valor), decimales)
    except (TypeError, ValueError):
        return valor


def _faltas_de_figura(figura, tipo, con_ejes, minimo_series, titulo_minimo):
    """Que le falta a una figura de Plotly para estar completa."""
    if figura is None:
        return ["sin resolver todavia: la variable de la figura sigue valiendo None"]
    if not hasattr(figura, "data") or not hasattr(figura, "layout"):
        return ["eso no es una figura de Plotly: px.line / px.bar / px.box devuelven una, "
                "y hay que guardarla en la variable"]

    faltas = []
    trazas = list(figura.data)
    if len(trazas) < minimo_series:
        faltas.append(f"la figura tiene {len(trazas)} serie(s) dibujada(s) y se esperan al menos "
                      f"{minimo_series}. Si esperaba varias, falta el argumento color=")

    tipos_por_nombre = {"linea": "scatter", "barras": "bar", "caja": "box",
                        "dispersion": "scatter", "histograma": "histogram"}
    if tipo is not None:
        esperado = tipos_por_nombre[tipo]
        reales = {traza.type for traza in trazas}
        if esperado not in reales:
            faltas.append(f"se pedia un grafico de {tipo} ({esperado}) y la figura trae "
                          f"{sorted(reales) or 'nada'}. Revise que llamo a la funcion px correcta")
        if tipo == "linea":
            modos = {getattr(traza, "mode", None) or "lines" for traza in trazas
                     if traza.type == "scatter"}
            if not any("lines" in (modo or "") for modo in modos):
                faltas.append("la figura es de puntos, no de linea. px.line une los puntos; "
                              "px.scatter no")

    titulo = (figura.layout.title.text or "").strip()
    if not titulo:
        faltas.append("falta el titulo: es el argumento title= de px, o "
                      "fig.update_layout(title=...)")
    elif len(titulo) < titulo_minimo:
        faltas.append(f"el titulo tiene {len(titulo)} caracteres y se piden al menos "
                      f"{titulo_minimo}. Un titulo que dice una conclusion no cabe en dos "
                      f"palabras: 'Consumo por municipio' es una etiqueta, no un mensaje")

    if con_ejes:
        eje_x = (figura.layout.xaxis.title.text or "").strip()
        eje_y = (figura.layout.yaxis.title.text or "").strip()
        if not eje_x and not eje_y:
            faltas.append("los dos ejes estan sin etiquetar. El eje de la medida necesita nombre "
                          "y unidad: fig.update_layout(yaxis_title='Promedio (ug/m3)') si la "
                          "medida va en el eje y, o xaxis_title si el grafico es horizontal")
    return faltas

def _faltas_de_kpis(funcion, df, minimo):
    """Que le falta a la funcion de KPIs: tres numeros y que reaccionen a los filtros."""
    if funcion is None:
        return ["sin resolver todavia: la funcion sigue valiendo None"]
    if not callable(funcion):
        return ["eso no es una funcion. Se espera def calcular_kpis(df): ... return {...}"]
    if not isinstance(df, pd.DataFrame):
        return ["todavia no hay DataFrame sobre el cual probar la funcion: resuelva primero la "
                "tarea que carga los datos de su equipo"]

    faltas = []
    try:
        completo = funcion(df)
    except Exception as error:  # noqa: BLE001 - el mensaje del error es el contenido
        return [f"la funcion revento al llamarla con el DataFrame completo: "
                f"{type(error).__name__}: {error}"]

    if not isinstance(completo, dict):
        return ["la funcion tiene que devolver un diccionario {'nombre del KPI': numero}"]
    if len(completo) < minimo:
        faltas.append(f"la funcion devuelve {len(completo)} KPI(s) y se piden al menos {minimo}")

    for nombre, valor in completo.items():
        if not str(nombre).strip():
            faltas.append("hay un KPI sin nombre. El nombre es lo que el usuario lee")
        try:
            float(valor)
        except (TypeError, ValueError):
            faltas.append(f"el KPI '{nombre}' no es un numero: {valor!r}. Formatee al mostrarlo, "
                          f"no al calcularlo")

    # La prueba que de verdad importa: un KPI que no cambia al filtrar es decoracion.
    mitad = df.iloc[: max(len(df) // 2, 1)]
    try:
        parcial = funcion(mitad)
    except Exception as error:  # noqa: BLE001
        faltas.append(f"la funcion revento con un subconjunto de filas: "
                      f"{type(error).__name__}: {error}. En la app va a recibir el DataFrame "
                      f"filtrado, que casi nunca son todas las filas")
        return faltas

    iguales = [nombre for nombre, valor in completo.items()
               if nombre in parcial and str(parcial[nombre]) == str(valor)]
    if completo and len(iguales) >= len(completo):
        faltas.append("ninguno de los KPIs cambio al recibir la mitad de las filas: estan "
                      "calculados sobre constantes o sobre el DataFrame completo, no sobre el "
                      "que entra por parametro. En la app eso se ve como tres numeros que no se "
                      "mueven nunca")
    return faltas

def _faltas_de_plan(plan, columnas, minimo):
    """Que le falta al plan de filtros: tres, distintos, uno categorico y uno de rango."""
    if plan is None:
        return ["sin resolver todavia: la variable del plan sigue valiendo None"]
    if not isinstance(plan, (list, tuple)) or not all(isinstance(f, dict) for f in plan):
        return ["se espera una lista de diccionarios, uno por filtro, con las claves "
                "'columna', 'widget' y 'porque'"]

    faltas = []
    if len(plan) < minimo:
        faltas.append(f"hay {len(plan)} filtro(s) y el minimo son {minimo}")
    if len(plan) > 5:
        faltas.append(f"hay {len(plan)} filtros y el techo son 5. La habilidad que se evalua "
                      f"es elegir")

    widgets = []
    usadas = []
    for numero, filtro in enumerate(plan, start=1):
        faltantes = [c for c in ("columna", "widget", "porque") if c not in filtro]
        if faltantes:
            faltas.append(f"al filtro {numero} le faltan las claves {faltantes}")
            continue
        columna = str(filtro["columna"])
        widget = str(filtro["widget"]).strip().lower()
        if columna not in columnas:
            faltas.append(f"el filtro {numero} usa la columna '{columna}', que no existe en su "
                          f"DataFrame. Las que hay: {list(columnas)[:8]}...")
        usadas.append(columna)
        if widget not in ("multiselect", "selectbox", "slider", "radio", "checkbox",
                          "date_input"):
            faltas.append(f"el filtro {numero} declara el widget '{widget}', que no es de "
                          f"Streamlit. Use multiselect, selectbox, slider, radio o date_input")
        widgets.append(widget)
        if len(str(filtro["porque"]).split()) < 4:
            faltas.append(f"el filtro {numero} no explica por que importa ese corte. Un filtro "
                          f"sin justificacion es un filtro que sobra")

    if len(set(usadas)) < len(usadas):
        faltas.append("hay dos filtros sobre la misma columna: cuentan como uno solo")
    if widgets and not any(w in ("multiselect", "selectbox", "radio") for w in widgets):
        faltas.append("no hay ningun filtro categorico (multiselect, selectbox o radio)")
    if widgets and not any(w in ("slider", "date_input") for w in widgets):
        faltas.append("no hay ningun filtro de rango (slider o date_input). Si su dataset no "
                      "tiene fecha, el rango puede ser numerico: st.slider sobre cualquier "
                      "columna continua")
    return faltas

def _faltas_de_datos(df, minimo_filas, minimo_columnas):
    """Que le falta al DataFrame del equipo para poder alimentar un dashboard."""
    if df is None:
        return ["sin resolver todavia: la variable sigue valiendo None"]
    if not isinstance(df, pd.DataFrame):
        return ["eso no es un DataFrame. pd.read_csv(...) devuelve uno"]

    faltas = []
    if len(df) < minimo_filas:
        faltas.append(f"el DataFrame tiene {len(df)} filas y se esperan al menos {minimo_filas}")
    if df.shape[1] < minimo_columnas:
        faltas.append(f"el DataFrame tiene {df.shape[1]} columnas y se esperan al menos "
                      f"{minimo_columnas}")

    sucias = [str(c) for c in df.columns
              if str(c) != str(c).lower() or " " in str(c)
              or any(letra in str(c) for letra in "áéíóúñÁÉÍÓÚÑ")]
    if sucias:
        faltas.append(f"estas columnas siguen con mayusculas, espacios o tildes: {sucias[:6]}. "
                      f"Renombre una sola vez, en la carga, y use ese nombre en el cuaderno y "
                      f"en la app. Un KeyError por una tilde a mitad del reto cuesta diez "
                      f"minutos que no tiene")

    numericas = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not numericas:
        faltas.append("no hay ninguna columna numerica: sin eso no hay KPI ni eje y. "
                      "pd.to_numeric(col, errors='coerce') convierte texto a numero")
    categoricas = [c for c in df.columns
                   if not pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() <= 60]
    if not categoricas:
        faltas.append("no hay ninguna columna categorica usable (texto con 60 valores "
                      "distintos o menos): sin eso no hay filtro categorico ni color=")
    return faltas


def _reportar(clave, faltas, frase_correcto):
    if faltas:
        print(f"[{clave}] Todavia no esta completo:")
        for falta in faltas:
            print(f"[{clave}]   - {falta}")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
    else:
        _RESULTADOS[clave] = True
        print(f"[{clave}] CORRECTO: {frase_correcto}")


def comprobar(clave, valor, decimales=None):
    """Dice si el resultado es el correcto, sin revelar cual era."""
    _RESULTADOS[clave] = False
    if valor is None:
        print(f"[{clave}] Sin resolver todavia: la variable sigue valiendo None.")
        return
    valor = _redondear(valor, decimales)
    if isinstance(valor, pd.DataFrame):
        print(f"[{clave}] Usted produjo un DataFrame de {valor.shape[0]} filas "
              f"y {valor.shape[1]} columnas.")
    elif isinstance(valor, pd.Series):
        print(f"[{clave}] Usted produjo una Series de {len(valor)} elementos.")
    else:
        print(f"[{clave}] Usted produjo: {valor!r}")
    if _huella(valor) == _ESPERADO.get(clave):
        _RESULTADOS[clave] = True
        print(f"[{clave}] CORRECTO.")
    else:
        print(f"[{clave}] Todavia no coincide.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")


def comprobar_figura(clave, figura, tipo=None, con_ejes=True, minimo_series=1,
                     titulo_minimo=25):
    """Revisa que la figura exista y cumpla lo que si es verificable.

    No hay una unica figura correcta, asi que lo que se revisa es lo que la clase 8
    dejo como no negociable: que este dibujada, que sea del tipo que responde la
    pregunta, que tenga titulo y que el eje de la medida diga su unidad. Si el
    titulo dice una conclusion VERDADERA no lo puede revisar ningun programa. Eso
    lo juzga usted, y es lo que mas pesa.
    """
    _RESULTADOS[clave] = False
    _reportar(clave, _faltas_de_figura(figura, tipo, con_ejes, minimo_series, titulo_minimo),
              "la figura cumple lo que se revisa aqui. Que el titulo sea verdadero lo juzga usted.")


def comprobar_kpis(clave, funcion, df, minimo=3):
    """Revisa que la funcion de KPIs devuelva numeros y que reaccionen a los filtros."""
    _RESULTADOS[clave] = False
    _reportar(clave, _faltas_de_kpis(funcion, df, minimo),
              "son numeros y cambian cuando cambian las filas. Eso es un KPI.")


def comprobar_plan(clave, plan, df, minimo=3):
    """Revisa el plan de filtros contra las columnas que su DataFrame tiene de verdad."""
    _RESULTADOS[clave] = False
    columnas = list(df.columns) if isinstance(df, pd.DataFrame) else []
    _reportar(clave, _faltas_de_plan(plan, columnas, minimo),
              "tres filtros distintos, uno categorico y uno de rango, sobre columnas que existen.")


def comprobar_datos(clave, df, minimo_filas=100, minimo_columnas=3):
    """Revisa que el DataFrame del equipo pueda alimentar un dashboard."""
    _RESULTADOS[clave] = False
    _reportar(clave, _faltas_de_datos(df, minimo_filas, minimo_columnas),
              "el DataFrame esta listo para alimentar la app.")


def resumen_puntos_de_control():
    """Estado de las diez tareas."""
    print("Punto de control")
    print("-" * 42)
    for clave in _CLAVES:
        estado = "correcto" if _RESULTADOS.get(clave) else "pendiente"
        print(f"  {clave}: {estado}")
    logrados = sum(1 for c in _CLAVES if _RESULTADOS.get(c))
    print("-" * 42)
    print(f"{logrados} de {len(_CLAVES)} {'correctas' if logrados != 1 else 'correcta'}.")


print("Verificador listo. Las tareas se comprueban con comprobar('T1', su_variable).")

## Tarea 1 · La serie nacional de deserción

**La pregunta.** ¿La deserción escolar del país está bajando o subiendo?

**El concepto.** Una serie de tiempo es una fila por periodo. Aquí hay una fila por departamento y
año, así que hay que colapsar los departamentos: `groupby('ano')` y promediar. Es el `groupby` de la
clase 4, sin nada nuevo.

**Ojo con lo que se promedia.** Este promedio trata a San Andrés igual que a Antioquia, porque no
está ponderado por número de estudiantes. Es una simplificación y hay que saber que se está haciendo.

**Los comandos.**

```python
df.groupby('columna', as_index=False)['medida'].mean()
tabla.sort_values('columna')
```

**Lo que decide usted.** Nada todavía: esta tarea tiene una sola respuesta. Se la damos como
calentamiento, porque las tres figuras salen de aquí.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Agrupe educacion por 'ano' (con as_index=False) y saque el promedio de 'desercion'.
# 2. Ordene por año, de menor a mayor.
# 3. Guarde la tabla en serie_anual.

serie_anual = None

comprobar("T1", serie_anual, decimales=4)

## Tarea 2 · El KPI con su ancla

**La pregunta.** En el último año disponible, ¿cuántos departamentos están por encima de la meta del
3%?

**El concepto.** Un KPI es un número que cambia una decisión. "Deserción promedio: 4,0%" no cambia
ninguna. "N departamentos incumplen la meta" sí: nombra el tamaño del problema. El número lo saca
usted en la celda de abajo; no lo escriba antes de calcularlo.

**Los comandos.**

```python
df['columna'].max()                 # el ultimo ano
df[df['columna'] == valor]          # el recorte
(serie > umbral).sum()              # el conteo de los que superan
int(...)                            # numpy devuelve su propio entero; conviertalo
```

**Lo que decide usted.** Nada del cálculo, pero sí esto: piense qué haría un ministro con ese número.
Si no se le ocurre nada, el KPI está mal elegido y en su dashboard va a pasar lo mismo.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Encuentre el último año disponible en educacion.
# 2. Recorte a las filas de ese año.
# 3. Cuente cuántas tienen 'desercion' por encima de META_DESERCION.
# 4. Guarde el conteo en departamentos_sobre_meta, como int.

departamentos_sobre_meta = None

comprobar("T2", departamentos_sobre_meta)

## Tarea 3 · Los diez departamentos con más deserción

**La pregunta.** ¿Dónde está peor el problema?

**El concepto.** Mismo `groupby`, otra columna de agrupación. Y `head(10)` **después** de ordenar:
si ordena después de recortar, se queda con diez departamentos cualesquiera bien ordenados, que es
un error que no lanza ningún mensaje.

**Los comandos.**

```python
df.groupby('columna', as_index=False)['medida'].mean()
tabla.sort_values('medida', ascending=False)
tabla.head(10)
```

**Lo que decide usted.** Nada del cálculo. Sí decide, en la tarea 5, cómo se muestra.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Agrupe educacion por 'departamento' y saque el promedio de 'desercion'.
# 2. Ordene de mayor a menor.
# 3. Quédese con los 10 primeros y guarde la tabla en top_departamentos.

top_departamentos = None

comprobar("T3", top_departamentos, decimales=4)

## Tarea 4 · La figura de línea

**La pregunta.** ¿Cómo se ve la evolución de la tarea 1?

**El concepto.** `px.line` une los puntos, y unir puntos afirma que entre uno y otro hay continuidad.
Con años eso es cierto. Con categorías (departamentos, estratos, municipios) **no lo es**, y por eso
la regla 3 de la clase 8 prohíbe la línea sobre categorías.

**Los comandos.**

```python
fig = px.line(tabla, x='columna', y='medida', title='...')
fig.update_layout(xaxis_title='', yaxis_title='Unidad')
fig.add_hline(y=valor, line_dash='dash')     # la linea de meta, opcional
fig.show()
```

**Lo que decide usted.** El título. El verificador exige que tenga al menos 25 caracteres, no por
capricho: un título de dos palabras es una etiqueta, y una etiqueta repite lo que los ejes ya dicen.
Mírela primero, escriba la conclusión que ve, y **verifíquela contra `serie_anual`** antes de darla
por buena.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Construya con px.line una figura sobre serie_anual (x: el año, y: la deserción).
# 2. Póngale un título que diga una conclusión, y la etiqueta del eje y con su unidad.
# 3. Guárdela en fig_linea y llame a .show()

fig_linea = None

comprobar_figura("T4", fig_linea, tipo="linea")

## Tarea 5 · La figura de barras

**La pregunta.** ¿Cuáles son los diez departamentos con más deserción, y qué tan lejos están del
resto?

**El concepto.** Barras para comparar categorías, ordenadas por valor (regla 4 de la clase 8). Con
nombres largos, las barras **horizontales** se leen mejor: `orientation='h'` o `px.bar(x=medida,
y=categoria)`. Y ojo: Plotly ordena el eje categórico como le parece, así que el orden de la tabla no
se respeta solo.

**Los comandos.**

```python
fig = px.bar(tabla, x='medida', y='categoria', orientation='h', title='...')
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.update_layout(xaxis_title='Unidad', yaxis_title='')
```

**Lo que decide usted.** La orientación, el orden y el título.

**Antes de escribir el título, imprima `top_departamentos` y léalo.** El error más caro de esta
clase no es de código: es titular de memoria. Si escribe "el departamento X duplica la meta" sin
haber mirado la cifra de X, tiene un 50% de probabilidad de estar afirmando algo falso con toda
seguridad tipográfica. Y va a ser lo primero que alguien note en la sustentación.

**Y mire la tabla con cuidado.** La celda de carga ya estandarizó `departamento` con la receta de la
clase 3, tildes incluidas. Sin ese paso, `Guainia` (una sola fila, con un 10,94% que no representa a
nadie) y `Guainía` (trece filas, 6,62%) habrían salido como dos barras distintas, y la más alta del
gráfico no habría sido el departamento con más deserción sino un artefacto de escritura. **Un
gráfico no arregla datos sucios: los publica.** Antes de titular, mire que las 10 categorías sean 10
departamentos distintos.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Construya con px.bar una figura sobre top_departamentos.
# 2. Ordene las barras y etiquete el eje de la medida con su unidad.
# 3. Póngale un título que diga una conclusión. Guárdela en fig_barras y llame a .show()

fig_barras = None

comprobar_figura("T5", fig_barras, tipo="barras")

## Tarea 6 · La figura de caja

**La pregunta.** El promedio nacional de la tarea 1, ¿esconde departamentos muy distintos entre sí?

**El concepto.** La caja muestra la **dispersión** que el promedio aplasta. Una caja por año revela
si los departamentos se están pareciendo entre ellos (cajas que se encogen) o separando (cajas que
crecen). Dos años con el mismo promedio y cajas distintas son dos países distintos.

Es la jirafa de la clase 4, otra vez: el promedio del parque no le sirve a nadie si hay una jirafa.

**Los comandos.**

```python
fig = px.box(df, x='categoria', y='medida', title='...')
fig.update_layout(xaxis_title='', yaxis_title='Unidad')
```

**Lo que decide usted.** Qué va en el eje x. Hay dos opciones defendibles (el año, o el
departamento), y responden preguntas distintas. Elija una y **escriba abajo por qué**.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Construya con px.box una figura sobre educacion.
# 2. Etiquete el eje de la medida y póngale un título con una conclusión.
# 3. Guárdela en fig_caja y llame a .show()

fig_caja = None

comprobar_figura("T6", fig_caja, tipo="caja")

**Tu respuesta.** ¿Qué eligió en el eje x de la caja, y qué pregunta responde esa versión que
la otra no responde?

*Tu respuesta:*

**Tu respuesta.** Las tres figuras de la parte A, ¿responden tres preguntas distintas o tres
versiones de la misma? Justifique en una línea por figura.

*Tu respuesta:*

---

# Parte B · Su dashboard, con el dataset de su equipo

De aquí en adelante no hay respuesta correcta que un programa pueda comparar: cada equipo trae un
archivo distinto. Lo que se comprueba es la **estructura**, que es justo lo que la app necesita para
funcionar.

## Paso 0 · Diseño en papel

**Sin código. Diez minutos.** Si no puede responder la primera pregunta, todavía no tiene un
dashboard: tiene un dataset.

El orden es este, y es el inverso al que todo el mundo usa:

| Paso | Qué se decide | Error típico |
|------|---------------|--------------|
| 1. Pregunta | Qué quiere saber el usuario | Saltárselo y empezar por el gráfico bonito |
| 2. Filtros | Qué cortes de esa pregunta importan | Poner un filtro por cada columna del dataset |
| 3. Métricas | Qué tres números resumen la respuesta | Poner el número fácil de calcular en vez del que importa |
| 4. Gráficos | Qué muestra lo que los números no alcanzan a decir | Meter el gráfico primero y buscarle la pregunta después |

**Tu respuesta.**

### La pregunta

- **¿Qué quiere saber quién usaría esto?** (una pregunta, no un tema)

- **¿Quién es esa persona?** (un cargo concreto, no "los usuarios")

- **¿Qué haría distinto después de mirarlo?**

### Los 3 gráficos

| # | Tipo | Qué pregunta responde |
|---|------|----------------------|
| 1 | | |
| 2 | | |
| 3 | | |

Tres gráficos de la misma cosa cuentan como uno. **Techo: más de cinco filtros o más de cinco
gráficos se penaliza.** La habilidad que se evalúa es elegir.

## Tarea 7 · Su dataset, cargado y renombrado

**La pregunta.** ¿Su archivo está en condiciones de alimentar una app?

**El concepto.** Renombrar las columnas **una sola vez, en la carga**, a minúsculas sin tildes ni
espacios. No es cosmético: en el `.py` de la app va a escribir esos nombres veinte veces, y un
`KeyError` por una tilde a mitad del reto cuesta diez minutos que no tiene. Es el atasco número uno
de esta clase.

Y después, los tres tipos: fecha con `pd.to_datetime`, medida con `pd.to_numeric`, y los valores
imposibles fuera (con el conteo escrito, como en el demo).

**Los comandos.**

```python
df = pd.read_csv('ruta')
df = df.rename(columns={'Nombre Original': 'nombre_corto'})
df['fecha'] = pd.to_datetime(df['fecha'], format='...', errors='coerce')
df['valor'] = pd.to_numeric(df['valor'], errors='coerce')
df = df.dropna(subset=['valor'])
```

**Lo que decide usted.** Todo: qué archivo, qué columnas conserva, cómo las llama y qué descarta.
La única regla dura es que quede escrito **cuántas filas descartó y por qué**.

**Si su dataset no está listo hoy** (no lo trajo, no abre, no tiene columnas usables): use el de
calidad del aire del demo, `../datos/calidad_aire_risaralda.csv`, para no perder el bloque, y traiga
el suyo a la clase 11, que es el laboratorio de ensayo. Lo que no puede hacer es entregar el Momento
2 sobre el dataset del aire: ese es del demo.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Cargue el CSV de su equipo.
# 2. Renombre las columnas que va a usar: minúsculas, sin tildes ni espacios.
# 3. Convierta los tipos (fecha y numéricas) y descarte los valores imposibles.
# 4. Imprima cuántas filas quedaron y cuántas descartó.
# 5. Guarde el resultado en df_equipo.

RUTA_EQUIPO = "../datos/tu_archivo.csv"   # <- cámbiela por la de su equipo

df_equipo = None

comprobar_datos("T7", df_equipo)

## Tarea 8 · El plan de filtros

**La pregunta.** ¿Cuáles son los tres cortes que su usuario de verdad necesita?

**El concepto.** Los filtros no son los que el dataset permite construir: son los que esa persona
usa. Y hay dos reglas duras, que el verificador revisa:

- **Al menos uno categórico** (`multiselect`, `selectbox` o `radio`) y **al menos uno de rango**
  (`slider` o `date_input`). Si su dataset no tiene fecha, el de rango puede ser numérico: un
  `st.slider` sobre cualquier columna continua.
- **Los tres sobre columnas distintas.** Dos filtros sobre columnas redundantes (municipio y
  estación, cuando cada municipio tiene una sola estación) cuentan como **uno**.

**Los comandos.** Aquí no se escribe la app todavía: se declara el plan como una lista de
diccionarios, para poder revisarlo antes de gastar veinte minutos programándolo.

```python
plan_filtros = [
    {'columna': 'nombre_de_columna', 'widget': 'multiselect', 'porque': 'una frase'},
]
```

**Lo que decide usted.** Los tres filtros y su justificación. La clave `'porque'` no es relleno: un
filtro que no se puede justificar en una frase es un filtro que sobra, y en la sustentación se
pregunta.

In [ ]:
# TU CÓDIGO AQUÍ
# Declare sus tres filtros. La columna tiene que existir en df_equipo con ese nombre exacto.
# Widgets válidos: multiselect, selectbox, radio, slider, date_input.

plan_filtros = None

comprobar_plan("T8", plan_filtros, df_equipo)

## Tarea 9 · La función de KPIs

**La pregunta.** ¿Qué tres números resumen la respuesta, y cambian cuando el usuario mueve los
filtros?

**El concepto.** En la app, los KPIs se calculan sobre el DataFrame **ya filtrado**. Por eso se
escriben dentro de una función que recibe el DataFrame por parámetro: así el mismo código sirve para
la vista completa y para cualquier recorte.

**Este es el error que el verificador caza:** calcular el KPI usando `df_equipo` por fuera de la
función, en vez del parámetro. La app arranca, se ve bien, y el número **no se mueve nunca**. Un KPI
que no reacciona a los filtros es decoración. El verificador llama a su función dos veces, con todas
las filas y con la mitad, y compara.

**Los comandos.**

```python
def calcular_kpis(df):
    return {'nombre del KPI': numero, ...}

len(df)                       # un conteo es un KPI valido
df['columna'].nunique()       # cuantas categorias distintas hay en la vista
(df['columna'] > umbral).mean() * 100     # un porcentaje sobre un umbral conocido
```

**Lo que decide usted.** Los tres números y sus nombres. Un máximo, un conteo, un porcentaje sobre un
umbral o una diferencia suelen decir más que un promedio.

In [ ]:
# TU CÓDIGO AQUÍ
# Escriba calcular_kpis(df): recibe un DataFrame y devuelve un diccionario con TRES KPIs.
# Todo se calcula a partir del parámetro df. Nada de df_equipo adentro.

def calcular_kpis(df):
    return {}


comprobar_kpis("T9", calcular_kpis, df_equipo)

## Tarea 10 · Las tres figuras — sin el orden

**La pregunta.** ¿Cuáles son las tres figuras de su dashboard?

Esta es la tarea que más pesa, y la única donde los comandos vienen **sin el orden**. Todo lo que
necesita apareció en el demo y en la parte A. Ármelo usted.

```python
fig.update_layout(xaxis_title='', yaxis_title='Unidad')
df.groupby('columna', as_index=False)['medida'].mean()
px.box(df, x='categoria', y='medida', title='...')
figuras_equipo = [fig_1, fig_2, fig_3]
tabla.sort_values('medida', ascending=False)
px.line(tabla, x='periodo', y='medida', color='serie', title='...')
fig.show()
px.bar(tabla, x='categoria', y='medida', title='...')
pd.Grouper(key='fecha', freq='ME')
```

**Lo que decide usted.** Todo: qué tres preguntas, qué tipo de gráfico responde cada una, qué se
agrega antes de graficar y qué dicen los tres títulos.

**Los tres criterios que se revisan:** que las tres figuras existan y estén dibujadas, que tengan
título de al menos 25 caracteres y etiqueta en el eje de la medida, y que los tres títulos sean
**distintos entre sí**. Que respondan tres preguntas distintas de verdad lo juzga usted.

**Recuerde:** `px` no agrega. El `groupby` va antes, siempre.

In [ ]:
# TU CÓDIGO AQUÍ
# Construya sus tres figuras sobre df_equipo y déjelas en la lista figuras_equipo.
# Agregue con groupby antes de graficar. Cada figura con su título-mensaje y su eje etiquetado.

figuras_equipo = None

comprobar_figura("T10", figuras_equipo[0] if figuras_equipo else None)

### Revisión de las tres figuras juntas

La celda de abajo ya está escrita: revisa las tres, una por una, y verifica que los títulos sean
distintos entre sí.

In [ ]:
titulos = []
if figuras_equipo:
    for numero, figura in enumerate(figuras_equipo, start=1):
        faltas = _faltas_de_figura(figura, None, True, 1, 25)
        estado = "completa" if not faltas else "; ".join(faltas)
        titulo = (figura.layout.title.text or "").strip() if hasattr(figura, "layout") else ""
        titulos.append(titulo)
        print(f"Figura {numero}: {estado}")
    if len(set(titulos)) < len(titulos):
        print("AVISO: hay dos figuras con el mismo título. Tres gráficos que dicen lo mismo "
              "cuentan como uno.")
else:
    print("Todavía no hay figuras: figuras_equipo sigue en None.")

In [ ]:
resumen_puntos_de_control()

---

# Paso final · De este cuaderno a la app

Ahora sí, abra `streamlit_app_starter.py`. Todo lo que probó aquí se muda casi tal cual:

| En este cuaderno | En la app |
|------------------|-----------|
| `RUTA_EQUIPO` y el `rename` de la tarea 7 | El bloque `CONFIGURACION` y la función `cargar_datos()` |
| `plan_filtros` de la tarea 8 | Los widgets del `st.sidebar` |
| `calcular_kpis(df)` de la tarea 9 | Los tres `st.metric`, sobre `df_filtrado` |
| Las figuras de la tarea 10 | `st.plotly_chart(fig, width="stretch")` |
| `fig.show()` | Nada: en la app lo reemplaza `st.plotly_chart` |

Desde una terminal, parada en `clase10/reto/`:

```
streamlit run streamlit_app_starter.py
```

Y para detenerla: `Ctrl+C` **en la terminal**. Cerrar la pestaña del navegador no la apaga.

## Antes de decir que terminó

1. **Kernel → Restart and Run All** en este cuaderno. Si algo revienta, arréglelo: un cuaderno que no
   corre de arriba a abajo le pone techo a la dimensión Hacer.
2. Ejecute el punto de control y revise las diez tareas.
3. Verifique que **todas** las celdas *Tu respuesta:* están escritas. La figura no es el análisis.
4. Relea los títulos de sus tres figuras y compruebe, uno por uno, que son **verdaderos** contra sus
   propias tablas.
5. La app levanta, los tres KPIs cambian al mover los filtros, y no revienta si deselecciona todo.

## Qué queda para la casa

- Revisar que los tres KPIs sean los que importan, no los que fueron fáciles de calcular.
- Reescribir los tres títulos como mensajes.
- Probar la app con filtros extremos: todo seleccionado, nada seleccionado, un solo valor.

**Opcional y no se evalúa:** `st.tabs`, `st.download_button`, temas y colores, mapas con
`px.scatter_map`, y el despliegue en Streamlit Community Cloud. **El despliegue no es requisito del
curso**: para la sustentación del Momento 2 basta con la app corriendo en local.

La clase 11 es el laboratorio de ensayo. Llegue con la app funcionando.